# Cyberbullying Detector - BERT Fine-tuning
**6-class classification:** clean, cyberbullying, harassment, hate_speech, threat, religious_hate

In [ ]:
!pip install -q transformers datasets pandas scikit-learn torch accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## 1. Upload your dataset
Click the folder icon on the left, then upload **merged_dataset_v3.csv**

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

LABELS = {0: 'clean', 1: 'cyberbullying', 2: 'harassment', 3: 'hate_speech', 4: 'threat', 5: 'religious_hate'}
label_names = [LABELS[i] for i in sorted(LABELS)]
num_labels = len(label_names)

print(f'Rows: {len(df)}')
print(f'Distribution:\n{df["label"].value_counts().sort_index()}')

In [ ]:
# Balance the dataset (cap each class at 8000, upsample threat)
MAX_PER_CLASS = 8000
balanced = []
for label_id in sorted(df['label'].unique()):
    subset = df[df['label'] == label_id]
    if len(subset) > MAX_PER_CLASS:
        subset = subset.sample(n=MAX_PER_CLASS, random_state=42)
    elif len(subset) < 500:
        n_needed = min(2000, MAX_PER_CLASS)
        dupes = subset.sample(n=n_needed - len(subset), replace=True, random_state=42)
        subset = pd.concat([subset, dupes])
    balanced.append(subset)
df = pd.concat(balanced, ignore_index=True)
print(f'Balanced: {len(df)} rows')
print(df['label'].value_counts().sort_index())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].astype(str).tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = ToxicDataset(X_train, y_train, tokenizer)
test_ds = ToxicDataset(X_test, y_test, tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

## 2. Train
This takes **30-60 min** on the free T4 GPU.

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    return {'eval_accuracy': acc, 'eval_f1_macro': f1_macro}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training...')
trainer.train()

## 3. Results

In [ ]:
preds = trainer.predict(test_ds)
y_pred = np.argmax(preds.predictions, axis=1)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

## 4. Download the model
Extract this zip into `backend/models/saved_models/`

In [ ]:
model.save_pretrained('cyberbullying_bert')
tokenizer.save_pretrained('cyberbullying_bert')
!zip -r cyberbullying_bert.zip cyberbullying_bert
files.download('cyberbullying_bert.zip')
print('Done! Upload this file and I will update the classifier to use it.')